# Extract Data from DKG Ontology
Make sure you have all packages installed before running the code. `pip install`

In [1]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase, RoutingControl, Result
import pandas as pd

## Connect to ontology db

In [2]:
# load secrets from .env file
load_dotenv()
onto_reader = os.getenv("ONTO_READER")
onto_reader_password = os.getenv("ONTO_READER_PASSWORD")
onto_url = os.getenv("ONTO_URL")

In [ ]:
# Define and check connection of DKG
url = onto_url
user = onto_reader # your username
password = onto_reader_password # your password
db_name = db_name # Your database name
driver = GraphDatabase.driver(url, auth=(user, password))

driver.verify_connectivity()

## Extract metadata sources and asset areas

In [4]:
# get data from the graph and store them in a dataframe
with driver.session() as session:
    query = (
            '''
            match (n:MetadataSource)-[:IS_SOURCE_OF]->(m)
            RETURN distinct n.display_name as MetadataSource, m.display_name as AssetType
            order by MetadataSource, AssetType
            '''
            )
    result = session.run(query)
    records = [record for record in result]
    metadata_sources = pd.DataFrame(records, columns=['MetadataSource', 'AssetType'])    

## Extract relationship types between metadata asset types

### Tested

In [ ]:
# get data from the graph and store them in a dataframe
with driver.session() as session:
    query = (
            '''
            match p=(n:AssetType)-[r]->(m:AssetType)
            where n.status='tested' and m.status='tested'
            return n.name as SourceType, r.name as RelationType, m.name as TargetType, 
                   n.id as SourceTypeId, r.id as RelationTypeId, m.id as TargetTypeId
            order by SourceType;
            '''
    )
    result = session.run(query)
    records = [record for record in result]
    relations_tested = pd.DataFrame(records, columns=['SourceType', 'RelationType', 'TargetType', 
                                                      'SourceTypeId', 'RelationTypeId', 'TargetTypeId'])

### Candidate

In [6]:
# get data from the graph and store them in a dataframe
with driver.session() as session:
    query = (
            '''
            match p=(n:AssetType)-[r]->(m:AssetType)
            where n.status='candidate' and m.status='candidate'
            return n.name as SourceType, r.name as RelationType, m.name as TargetType 
            order by SourceType;
            '''
    )
    result = session.run(query)
    records = [record for record in result]
    relations_candidate = pd.DataFrame(records, columns=['SourceType', 'RelationType', 'TargetType'])

## Extract tested asset types and their asset areas

In [ ]:
# get data from the graph and store them in a dataframe
with driver.session() as session:
    query = (
            '''
            match p=(n:AssetType)-[r]->(m:AssetArea)-[]-(aat:AssetAreaType)
            where n.status='tested'
            return n.name as AssetType, m.name as AssetArea, aat.name as AssetAreaType, 
                   n.id as AssetTypeId, m.id as AssetAreaId, aat.id as AssetAreaTypeId
            order by AssetAreaType, AssetArea, AssetType
            '''
            )
    result = session.run(query)
    records = [record for record in result]
    asset_areas = pd.DataFrame(records, columns=['AssetType', 'AssetArea', 'AssetAreaType', 
                                                 'AssetTypeId', 'AssetAreaId', 'AssetAreaTypeId'])

## Extract tested asset types and their attribute types

In [8]:
# get data from the graph and store them in a dataframe
with driver.session() as session:
    query = (
            '''
            match p=(n:AssetType)-[r]->(m:AttributeType)
            where n.status='tested'
            return n.name as AssetType, m.name as AttributeType, n.id as AssetTypeId, m.id as AttributeTypeId
            order by AssetType, AttributeType
            '''
            )
    result = session.run(query)
    records = [record for record in result]
    attributes = pd.DataFrame(records, columns=['AssetType', 'AttributeType', 'AssetTypeId', 'AttributeTypeId'])

## Extract schemas

In [9]:
# get data from the graph and store them in a dataframe
def extract_schema(label):
    with driver.session() as session:
        query = (
                f'MATCH (n:{label}) RETURN n;'
        )
        result = session.run(query)
        records = [record["n"] for record in result]
        return pd.DataFrame([dict(records[0])])

In [10]:
asset_schema = extract_schema('AssetSchema')
attribute_schema = extract_schema('AttributeSchema')
asset_area_schema = extract_schema('AssetAreaSchema')
relation_schema = extract_schema('RelationSchema')
metadata_source_schema = extract_schema('MetadataSourceSchema')